# Nuclear vs Wind/Solar: cleanliness rebuttal

This notebook tests the claim that **nuclear electricity is cleaner than wind or solar**.

It has two parts:

1. **Carbon-only** lifecycle greenhouse-gas emissions (gCO2e/kWh) with an uncertainty simulation. This is the most common metric behind the claim, and it is enough to reject the broad version.
2. **Beyond carbon** — a twelve-metric cleanliness profile with explicit uncertainty ranges, policy scenarios, Monte Carlo propagation and rank-stability diagnostics. This shows that *which* technology is “cleanest” depends on what you weight.

Metric definitions live in [`../DATA_DICTIONARY.md`](../DATA_DICTIONARY.md); data provenance in [`../data/data_sources.md`](../data/data_sources.md); the method summary in [`../docs/methods_note.md`](../docs/methods_note.md).

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from energy_cleanliness.data import load_lifecycle_data
from energy_cleanliness.analysis import (
    compare_to_baseline,
    simulate_uncertainty,
    estimate_pairwise_probabilities,
)
from energy_cleanliness.multimetric import (
    METRIC_UNITS,
    load_multimetric_profile,
    higher_is_better_set,
    to_wide,
)
from energy_cleanliness.scenarios import SCENARIOS
from energy_cleanliness.cleanliness_index import (
    monte_carlo_cleanliness,
    pareto_frontier,
)
from energy_cleanliness.reporting import build_report, validate_report
from energy_cleanliness.plotting import (
    plot_lifecycle_ranges,
    plot_probability_matrix,
    plot_scenario_scores,
    plot_rank_stability,
)

reports_dir = PROJECT_ROOT / 'reports'
figures_dir = reports_dir / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
SEED = 42
SAMPLES = 2000

## 1. Carbon-only lifecycle emissions

If the claim were broadly true, nuclear should have lower lifecycle emissions than wind and solar under the same source (IPCC AR5 Annex III).

In [ ]:
data = load_lifecycle_data(PROJECT_ROOT / 'data' / 'lifecycle_emissions_ipcc_ar5.csv')
data[['technology', 'min_gco2e_kwh', 'median_gco2e_kwh', 'max_gco2e_kwh']]

In [ ]:
comparison = compare_to_baseline(data, baseline='Nuclear')
comparison[['technology', 'baseline_minus_other', 'interpretation']]

Onshore wind has a **lower** median than nuclear and offshore wind is tied, so the broad “cleaner than wind” claim already fails on its own preferred metric.

### Uncertainty simulation

The reported min / median / max are turned into a triangular proxy distribution to make the overlap visible. This is a sensitivity check, not a reconstruction of the literature distribution. Cell `(A, B)` is the estimated probability that A has lower lifecycle emissions than B.

In [ ]:
samples = simulate_uncertainty(data, draws=50_000, seed=SEED)
probability_matrix = estimate_pairwise_probabilities(samples)
probability_matrix.round(3)

In [ ]:
plot_lifecycle_ranges(data, reports_dir / 'lifecycle_ranges.png')
plot_probability_matrix(probability_matrix, reports_dir / 'probability_matrix.png')
display(Image(filename=str(reports_dir / 'lifecycle_ranges.png')))
display(Image(filename=str(reports_dir / 'probability_matrix.png')))

## 2. Beyond carbon — multi-metric cleanliness

“Clean” is undefined in the claim. We make it explicit with twelve metrics, each carrying a `low / central / high` uncertainty range and a declared direction (`lower_better` or `higher_better`). The loader validates the schema and fails fast on misaligned columns or missing cells.

In [ ]:
profile = load_multimetric_profile(PROJECT_ROOT / 'data' / 'multimetric_cleanliness_reference.csv')
higher = higher_is_better_set(profile)
wide = to_wide(profile, value='central')
print('higher-is-better metrics:', sorted(higher))
profile.head(6)

In [ ]:
# Best technology per metric, respecting each metric's direction.
best_by_metric = {}
for metric in METRIC_UNITS:
    if metric in higher:
        best_by_metric[metric] = wide.loc[wide[metric].idxmax(), 'technology']
    else:
        best_by_metric[metric] = wide.loc[wide[metric].idxmin(), 'technology']
pd.Series(best_by_metric, name='best_technology').to_frame()

No single technology wins every metric. The **Pareto frontier** is the set that is never beaten on all metrics at once.

In [ ]:
frontier = pareto_frontier(
    wide, metrics=list(METRIC_UNITS), minimize=False, higher_is_better=higher
)
list(frontier['technology'])

### Policy scenarios with Monte Carlo uncertainty

A scenario is a set of metric weights encoding a *policy intent*. For each one we sample every metric from `triangular(low, central, high)`, normalise within the draw, weight and sum, then report the mean score, a 95% credible interval, and **rank stability** — how often each technology lands at rank 1.

In [ ]:
scenario_results = {}
rows = []
for name, weights in SCENARIOS.items():
    mc = monte_carlo_cleanliness(profile, weights=weights, samples=SAMPLES, seed=SEED)
    scenario_results[name] = mc
    summary = mc['score_summary']
    stability = mc['rank_stability']
    leader = summary.iloc[0]
    p_rank1 = float(stability.loc[stability['technology'] == leader['technology'], 'p_top1'].iloc[0])
    rows.append({
        'scenario': name,
        'leader': leader['technology'],
        'leader_mean_score': round(float(leader['mean_score']), 3),
        'leader_P(rank1)': round(p_rank1, 2),
        'ranking': ' > '.join(summary['technology']),
    })
leaders = pd.DataFrame(rows)
leaders

The leader **flips with the policy intent**: reliability-first favours nuclear, while cost-first and emissions-first favour wind. That is the whole point — there is no single “cleanest” technology.

Compare two contrasting scenarios visually — scores with 95% intervals, and rank stability.

In [ ]:
for scenario in ['high_reliability_first', 'low_emissions_first']:
    mc = scenario_results[scenario]
    score_png = figures_dir / f'scores_{scenario}.png'
    rank_png = figures_dir / f'rank_stability_{scenario}.png'
    plot_scenario_scores(mc['score_summary'], score_png, title=f'Scores — {scenario}')
    plot_rank_stability(mc['rank_stability'], rank_png, title=f'Rank stability — {scenario}')
    display(Image(filename=str(score_png)))
    display(Image(filename=str(rank_png)))

### Structured report

All of the above is assembled into a single versioned JSON document that a dashboard or API can consume without re-running the analysis (schema in [`../docs/report_schema.json`](../docs/report_schema.json)).

In [ ]:
report = build_report(
    dataset_schema_version='1.0',
    seed=SEED,
    samples=SAMPLES,
    scenarios={
        name: {
            'weights': SCENARIOS[name],
            'score_summary': result['score_summary'],
            'rank_stability': result['rank_stability'],
        }
        for name, result in scenario_results.items()
    },
    pareto_frontier=frontier,
    best_by_metric=best_by_metric,
)
validate_report(report)
{'top_level_keys': sorted(report), 'scenarios': sorted(report['scenarios']), 'method': report['method']}

## Conclusion

- On lifecycle **carbon**, the broad claim is wrong as written: onshore wind has a lower IPCC median than nuclear, offshore wind is tied, and the uncertainty ranges overlap.
- On the **wider** twelve-metric profile there is no universal winner. Nuclear is the most **reliable** (capacity factor, grid integration) and competitive on land and materials; wind leads on **cost** and **emissions**; solar PV is strong on **safety** and **construction time**.
- Rank-stability percentages and overlapping credible intervals show that single-number rankings overstate certainty.

> Nuclear, wind and solar are all low-carbon electricity sources. Nuclear is often lower-carbon than solar PV and is the most reliable, but it is not clearly cleaner than wind, and which option is “cleanest” depends on the metric and weighting choices.